# analysis.diversity

This notebook is aimed to compute:
- Alpha-Diversity according to different metrics (species richness and Chao1)
- Statistical differences between habitats and disturbance levels
- Post hoc analysis

Additionally, this notebook also creates the inputs for a correlation analysis carried out later.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from daforfer import DaforferDB
from skbio.diversity.alpha import chao1
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
db.toc()

In [ ]:
si = DaforferDB(conf['si'])
si.toc()

## PAB Diversity estimations


### Calculations

In [ ]:
alpha_diversity = db.conn.sql('SELECT * FROM D_Site_level_div').df()
alpha_diversity

### Species richness results

In [ ]:
g = sns.catplot(alpha_diversity, x='site', y='species_richness_bact', height=2.0, aspect=2.0, kind='bar', hue='habitat', palette=conf['habitat_palette'], edgecolor='black', linewidth=0.65)
# g.axes[0, 0].set_yscale('log')
g.set_ylabels("Bacteria species richness")
g.set_xlabels("Site code")
g.set_xticklabels(rotation=90)
g.savefig("figures/barplot.species-richness.colbyhabitat.svg")

In [ ]:
g = sns.catplot(
    alpha_diversity, x='disturbed', y='species_richness_bact', height=2.0, aspect=1.00, linewidth=0.65, kind='box', hue='disturbed', whis=20.0)
g.set_ylabels("Bacteria species richness")
g.set_xlabels("Disturbance level")
g.savefig("figures/boxplot.species-richness.bydisturbance.svg")

In [ ]:
g = sns.catplot(
    alpha_diversity, x='habitat', y='species_richness_bact', height=2.0, aspect=1.25, 
    hue='habitat', palette=conf['habitat_palette'], linewidth=0.65, kind='box', whis=20.0)
g.set_ylabels("Bacteria species richness")
g.set_xlabels("Habitat")
g.savefig("figures/boxplot.species-richness.byhabitat.svg")

### Chao1 diversity results 

In [ ]:
alpha_diversity

In [ ]:
g = sns.catplot(alpha_diversity, x='site', y='chao1_bact', height=2.0, aspect=2.0, kind='bar', hue='habitat', palette=conf['habitat_palette'], edgecolor='black', linewidth=0.65)
# g.axes[0, 0].set_yscale('log')
g.set_ylabels("Chao1 diversity index")
g.set_xlabels("Site code")
g.set_xticklabels(rotation=90)
g.savefig("figures/barplot.chao1.colbyhabitat.svg")

In [ ]:
g = sns.catplot(
    alpha_diversity, x='disturbed', y='chao1_bact', height=2.0, aspect=1.00, linewidth=0.65, kind='box', hue='disturbed', whis=20.0)
g.set_ylabels("Chao1 div. index")
g.set_xlabels("Disturbance level")
g.savefig("figures/boxplot.chao1.bydisturbance.svg")

In [ ]:
g = sns.catplot(
    alpha_diversity, x='habitat', y='chao1_bact', height=2.0, aspect=1.25, 
    hue='habitat', palette=conf['habitat_palette'], linewidth=0.65, kind='box', whis=200.0)
g.set_ylabels("Chao1 div. index")
g.set_xlabels("Habitat")
g.savefig("figures/boxplot.chao1.byhabitat.svg")

### Diversity by disturbance level

In [ ]:
q1_stats = []
for metric in ['species_richness_bact', 'chao1_bact']:
    kw_h, pval = stats.mannwhitneyu(
        alpha_diversity.query('habitat == "Crop" or habitat == "Edge"')[metric].values,
        alpha_diversity.query('habitat == "Wasteland" or habitat == "Oak"')[metric].values,
    )
    significative = pval < 0.05
    q1_stats.append(
        {'metric': metric, 'U': kw_h, 'p-val': pval, 'sign': significative}
    )
q1_stats = pd.DataFrame.from_records(q1_stats)

db.save_dataframe(
    df=q1_stats, table_name="T_ADDisturbance",
    description="Mann Whitney U test on species richness and Chao1 diversity index by disturbance level"
)
q1_stats

### Diversity by habitat

In [ ]:
q2_stats = []
for metric in ['species_richness_bact', 'chao1_bact']:
    kw_h, pval = stats.kruskal(
        alpha_diversity.query('habitat == "Crop"')[metric].values,
        alpha_diversity.query('habitat == "Edge"')[metric].values,
        alpha_diversity.query('habitat == "Wasteland"')[metric].values,
        alpha_diversity.query('habitat == "Oak"')[metric].values
    )
    significative = pval < 0.05
    q2_stats.append(
        {'metric': metric, 'H': kw_h, 'p-val': pval, 'sign': significative}
    )
q2_stats = pd.DataFrame.from_records(q2_stats)

db.save_dataframe(
    df=q2_stats, table_name="T_ADhabitat",
    description="Kruskal Wallis test on species richness and Chao1 diversity index by habitat"
)
q2_stats

### Diversity by habitat, post-hoc analysis

In [ ]:
q3_stats = []
for metric in ['species_richness_bact', 'chao1_bact']:
    for h1 in ['Crop', 'Edge', 'Wasteland', 'Oak']:
        for h2 in ['Crop', 'Edge', 'Wasteland', 'Oak']:
            if h1 != h2:
                kw_h, pval = stats.mannwhitneyu(
                    alpha_diversity.query('habitat == "{0}"'.format(h1))[metric].values,
                    alpha_diversity.query('habitat == "{0}"'.format(h2))[metric].values,
                    alternative='less'
                )
                significative = pval < 0.05
                q3_stats.append(
                    {'metric': metric, 'group_1': h1, 'group_2': h2, 'U': kw_h, 'p-val': pval, 'sign': significative}
                )
q3_stats_df = pd.DataFrame.from_records(q3_stats)

db.save_dataframe(
    df=q3_stats_df, table_name="T_ADHabitatPH",
    description="Post-Hoc Mann Whitney U analysis on species richness and Chao1 diversity index by habitat. Using group1 < group2 test"
)
si.save_dataframe(
    df=q3_stats_df, table_name="TableS5",
    description="Mann-Whitney U post-hoc test on site-diversity by habitat"
)
q3_stats_df

In [ ]:
q3_stats_df.query('metric == "species_richness_bact"').pivot(index='group_1', columns='group_2', values='p-val')

In [ ]:
q3_stats_df.query('metric == "species_richness_bact"').pivot(index='group_1', columns='group_2', values='U').round(4)

In [ ]:
q3_stats_df.query('metric == "chao1_bact"').pivot(index='group_1', columns='group_2', values='p-val').round(4)

In [ ]:
q3_stats_df.query('metric == "chao1_bact"').pivot(index='group_1', columns='group_2', values='U')

In [ ]:
db.conn.close()
si.conn.close()